# Lab: Cancer–Immune Cell Competition on a Lattice

**Course:** Modeling of Complex Biological Systems  
---

## Background

The interaction between a growing tumour and the immune system can be modelled as an
evolutionary game played on a spatial grid.  Each site of an $L\times L$ lattice is
occupied by either a **Cancer (C)** or an **Immune (I)** cell.  At every time step a
random cell competes with a random neighbour; the winner replaces the loser with a
probability that depends on their *fitness*, which itself depends on the local
neighbourhood composition.

### Mathematical formulation

| Symbol | Meaning |
|--------|---------|
| $L$ | grid side length ($L^2$ total sites) |
| $\mathcal{N}(i,j)$ | neighbourhood of site $(i,j)$ — von Neumann (4) or Moore (8) |
| $n_C, n_I$ | number of Cancer / Immune neighbours of a given cell |
| $a, b, c, d$ | payoff parameters (see below) |
| $\beta$ | selection intensity — controls how strongly fitness affects outcome |

**Fitness functions**

$$
f_C = a \cdot n_C + b \cdot n_I \qquad\qquad f_I = c \cdot n_C + d \cdot n_I
$$

**Competition (invasion) probability**

When cell $(i,j)$ competes with neighbour $(k,\ell)$, the probability that $(i,j)$
replaces $(k,\ell)$ is

$$
P = \frac{1}{1 + e^{-\beta\,(f_{(i,j)} - f_{(k,\ell)})}}
$$

This is a **Fermi function** — when $\beta=0$ the outcome is random (50 / 50); as
$\beta\to\infty$ the fitter cell always wins.

### Biological interpretation of payoff parameters

| Parameter | Meaning |
|-----------|---------|
| $a$ | Cancer–Cancer cooperation (growth-factor autocrine signalling) |
| $b$ | Cancer benefit when surrounded by Immune cells (immune evasion / exploitation) |
| $c$ | Immune effectiveness near Cancer cells (cytotoxic killing) |
| $d$ | Immune–Immune cooperation (co-stimulation, cytokine support) |

Three illustrative parameter sets:

| Scenario | $a$ | $b$ | $c$ | $d$ | Expected outcome |
|----------|-----|-----|-----|-----|------------------|
| Cancer advantage | 1.0 | 2.0 | 0.0 | 1.0 | Cancer wins |
| Coexistence | 0.8 | 1.2 | 0.5 | 1.5 | Spatial segregation |
| Immunotherapy | 0.6 | 0.5 | 2.0 | 1.8 | Immune wins |


## Setup

Run the cell below to import libraries and set up plotting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import animation
from IPython.display import HTML

# For reproducibility
np.random.seed(42)

# Custom colormap: Cancer = firebrick red, Immune = steelblue
CMAP = mcolors.ListedColormap(["firebrick", "steelblue"])
BOUNDS = [-0.5, 0.5, 1.5]
NORM = mcolors.BoundaryNorm(BOUNDS, CMAP.N)

def show_grid(grid, title="", ax=None):
    """Display the lattice.  Cancer (0) = red, Immune (1) = blue."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(grid, cmap=CMAP, norm=NORM, interpolation="nearest")
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    return ax


## Provided helper functions

The functions below implement the core simulation logic.  **Read them carefully** —
you will use (and extend) them in the exercises.


In [ ]:
# ---------------------------------------------------------------------------
#  Core simulation engine
# ---------------------------------------------------------------------------

def init_grid(L, frac_cancer=0.5):
    """
    Create an LxL grid.
    Each site is 0 (Cancer) with probability `frac_cancer`, else 1 (Immune).
    """
    return (np.random.rand(L, L) >= frac_cancer).astype(int)


def get_neighbours(i, j, L, mode="moore"):
    """
    Return list of neighbour coordinates for site (i, j) on an LxL grid.
    
    Parameters
    ----------
    mode : str
        "vonneumann" - 4 neighbours (up/down/left/right)
        "moore"      - 8 neighbours (includes diagonals)
    
    Boundary: sites outside the grid are simply excluded (no wrapping).
    """
    deltas_4 = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    deltas_8 = [(di, dj) for di in (-1, 0, 1) for dj in (-1, 0, 1)
                if (di, dj) != (0, 0)]
    deltas = deltas_4 if mode == "vonneumann" else deltas_8
    nbrs = []
    for di, dj in deltas:
        ni, nj = i + di, j + dj
        if 0 <= ni < L and 0 <= nj < L:
            nbrs.append((ni, nj))
    return nbrs


def compute_fitness(grid, i, j, a, b, c, d, mode="moore"):
    """
    Compute fitness of cell at (i, j).
    Returns f_C if the cell is Cancer, f_I if it is Immune.
    """
    L = grid.shape[0]
    nbrs = get_neighbours(i, j, L, mode)
    n_C = sum(1 for (ni, nj) in nbrs if grid[ni, nj] == 0)
    n_I = sum(1 for (ni, nj) in nbrs if grid[ni, nj] == 1)
    if grid[i, j] == 0:          # Cancer
        return a * n_C + b * n_I
    else:                         # Immune
        return c * n_C + d * n_I


def invasion_probability(f_attacker, f_defender, beta=1.0):
    """Fermi probability that the attacker replaces the defender."""
    return 1.0 / (1.0 + np.exp(-beta * (f_attacker - f_defender)))


def simulate(L, a, b, c, d, beta=1.0, frac_cancer=0.5,
             n_steps=None, mode="moore", record_every=None):
    """
    Run a full simulation until one cell type is eliminated or `n_steps`
    competition events have occurred.

    Parameters
    ----------
    L            : int   - grid side length
    a, b, c, d   : float - payoff parameters
    beta         : float - selection intensity
    frac_cancer  : float - initial fraction of Cancer cells
    n_steps      : int or None - max steps; if None defaults to 10 * L^2
    mode         : str   - "moore" or "vonneumann"
    record_every : int or None - if set, store a snapshot every this many steps

    Returns
    -------
    grid      : final grid state
    history   : dict with keys 'cancer_frac' (list of Cancer fractions),
                'steps_done' (int), and optionally 'snapshots' (list of grid copies)
    """
    grid = init_grid(L, frac_cancer)
    if n_steps is None:
        n_steps = 10 * L * L
    total = L * L

    history = {"cancer_frac": [], "steps_done": 0}
    if record_every is not None:
        history["snapshots"] = [grid.copy()]

    for step in range(1, n_steps + 1):
        # 1. Pick a random cell
        i, j = np.random.randint(0, L), np.random.randint(0, L)
        # 2. Pick a random neighbour
        nbrs = get_neighbours(i, j, L, mode)
        ki, kj = nbrs[np.random.randint(len(nbrs))]

        # Only compete if they are different types
        if grid[i, j] != grid[ki, kj]:
            f_ij = compute_fitness(grid, i, j, a, b, c, d, mode)
            f_kl = compute_fitness(grid, ki, kj, a, b, c, d, mode)
            p = invasion_probability(f_ij, f_kl, beta)
            if np.random.rand() < p:
                grid[ki, kj] = grid[i, j]   # (i,j) invades (k,l)
            else:
                grid[i, j] = grid[ki, kj]   # (k,l) invades (i,j)

        # Record fraction
        if step % max(1, n_steps // 200) == 0:
            cancer_count = np.sum(grid == 0)
            history["cancer_frac"].append(cancer_count / total)
        if record_every and step % record_every == 0:
            history["snapshots"].append(grid.copy())

        # Early stopping
        cancer_count_full = np.sum(grid == 0)
        if cancer_count_full == 0 or cancer_count_full == total:
            history["cancer_frac"].append(cancer_count_full / total)
            history["steps_done"] = step
            if record_every:
                history["snapshots"].append(grid.copy())
            break
    else:
        history["steps_done"] = n_steps

    return grid, history


### Quick demo — single run

In [ ]:
# Run one simulation with the "Coexistance" parameters
# TODO changed parameters 
grid_final, hist = simulate(L=30, a=1.0, b=2.0, c=0.0, d=1.0,
                            beta=1.0, n_steps=20000,
                            record_every=2000)

fig, axes = plt.subplots(1, min(len(hist["snapshots"]), 6), figsize=(18, 3))
for idx, ax in enumerate(axes):
    step = idx * 2000
    show_grid(hist["snapshots"][min(idx, len(hist["snapshots"])-1)],
              title=f"step {step}", ax=ax)
plt.suptitle("Cancer (red) vs Immune (blue) — Cancer-advantage scenario", y=1.02)
plt.tight_layout()
plt.show()


***

## Exercise 1 — Who wins? Monte Carlo outcome analysis

### Goal
Run many independent simulations for each of the three parameter sets and estimate
the **probability that Cancer takes over the grid**.

### Instructions

1. For each of the three scenarios below, run **50 simulations** (use $L=25$ for speed)
   and record whether Cancer won (final Cancer fraction = 1.0), Immune won (= 0.0),
   or neither dominated (time ran out).

| Scenario | $a$ | $b$ | $c$ | $d$ |
|----------|-----|-----|-----|-----|
| Cancer advantage | 1.0 | 2.0 | 0.0 | 1.0 |
| Coexistence | 0.8 | 1.2 | 0.5 | 1.5 |
| Immunotherapy | 0.6 | 0.5 | 2.0 | 1.8 |

2. Create a **grouped bar chart** showing the fraction of simulations won by Cancer,
   won by Immune, and unresolved, for each scenario.

3. For the *Coexistence* scenario, run **50 simulations with a fixed, limited
   time budget** (`n_steps = 20000`) — short enough that the system usually does
   **not** reach fixation.  Plot a **histogram** of the Cancer fraction at the end
   of these runs.  Is the distribution unimodal or bimodal?  What does this tell
   you about the dynamics?

**Hint:** you can call `simulate(...)` in a loop and inspect `np.mean(grid == 0)`
for the final cancer fraction.

> **Bonus:** How do the results change if you use `mode="vonneumann"` (4 neighbours)
> instead of `"moore"` (8 neighbours)?



### Answer

#### 3. Histogram of coexistence 
Plots shows ous bimodal (or rather trimodal) distribution with 3 peaks, which are:
- Cancer lose if fraction is equal to 0.0
- Cancer wins if fraction is equal to 1.0 
- Coexistence, where the center is around fraction equals to 0.6

It is caused by dynamics od system where we can see that changes in *fitness functions* are symmetric (by 0.3)  

To further examine dynamics we need to evaluate how it affects *fermi function*. We use the parameters for the Coexistence scenario ($a=0.8, b=1.2, c=0.5, d=1.5, \beta=1.0$). The fitness equations are:
* $f_C = 0.8 \cdot n_C + 1.2 \cdot n_I = (0.5 + 0.3) \cdot n_C + (1.5 - 0.3) \cdot n_I$
* $f_I = 0.5 \cdot n_C + 1.5 \cdot n_I$

This simplifies the difference in fitness to: 
$$f_I - f_C = 0.3 \cdot n_I - 0.3 \cdot n_C = 0.3 \cdot (n_I - n_C$$

To get an intuition about the formula, we calculate certain cases. Without loss of generality, we assume that the defender cell $(k,l)$ is Cancerous ($C$) and the attacking cell $(i,j)$ is Immune ($I$). The probability of the Immune cell replacing the Cancer cell is given by the Fermi function:$$P(I \rightarrow C) = \frac{1}{1 + \exp(-\beta(f_I - f_C))}$$

**Case 1**: $n_C = 4, n_I = 4$
$$\begin{align}f_I - f_C &= 0.3(4 - 4) = 0 \\mathbb{P}(I \rightarrow C) &= \frac{1}{1 + \exp(-1.0 \cdot 0)} = \frac{1}{1 + 1} = 0.500\end{align}$$
**Case 2**: $n_C = 5, n_I = 3$ (Advantage for Cancer)
$$\begin{align}f_I - f_C &= 0.3(3 - 5) = 0.3(-2) = -0.6 \\mathbb{P}(I \rightarrow C) &= \frac{1}{1 + \exp(-1.0 \cdot (-0.6))} = \frac{1}{1 + \exp(0.6)} \approx 0.354\end{align}$$
**Case 3**: $n_C = 6, n_I = 2$ (Strong Advantage for Cancer)
$$\begin{align}f_I - f_C &= 0.3(2 - 6) = 0.3(-4) = -1.2 \\mathbb{P}(I \rightarrow C) &= \frac{1}{1 + \exp(-1.0 \cdot (-1.2))} = \frac{1}{1 + \exp(1.2)} \approx 0.231\end{align}$$

Where reverse probabilities, where Cancer invades Immune, are simply $1 - \mathbb{P}(I \rightarrow C)$

We can conclude that this model prefers sparse families, which is biologically counter-intuitive. When immune cells are sparse, they cannot communicate with each other and should be dominated. Conversely, cancer cells should behave differently—with fewer cancer cells, the cancer can be more aggressive. The coefficients here do not work as intended, resulting in essentially only one winning strategy.

#### Comparison between moore and vonneumann mode
In *moore* process changes appear more quickly (more neighbors affect system), it is because when we have optimal scenario for cancer / immunity it just accelerates process. 

#### Comment 
Those are of course suggestions - we can state that because we do not posses information about how significant our results are. 

In [ ]:
# ─── Exercise 1: Your code here ───
from ast import Dict
import secrets
from matplotlib import axis
import polars as pl


scenarios = {
    "Cancer advantage": (1.0, 2.0, 0.0, 1.0),
    "Coexistence":      (0.8, 1.2, 0.5, 1.5),
    "Immunotherapy":    (0.6, 0.5, 2.0, 1.8),
}

N_SIMS = 50
L = 25
beta_value = 1.0
n_steps = 40_000  # enough for L=25

# TODO original - filling dict, i change tu df so i can contain all informations 
# results = {}  # fill this with outcome counts
results_moore = pl.DataFrame(
    {
        "scenario": [],
        "probe_n": [],
        "cancer_frac_history": []
    },
    schema={
        "scenario": pl.String,
        "probe_n": pl.Int64,
        "cancer_frac_history": pl.List(pl.Float64)
    }
)

results_vonneumann = pl.DataFrame(
    {
        "scenario": [],
        "probe_n": [],
        "cancer_frac_history": []
    },
    schema={
        "scenario": pl.String,
        "probe_n": pl.Int64,
        "cancer_frac_history": pl.List(pl.Float64)
    }
)

# --- YOUR CODE BELOW ---

## Simulation part
### mode: moore 
for name, (a, b, c, d) in scenarios.items():
    for probe_n in range(N_SIMS):
        _, hist = simulate(L= L, a=a, b=b, c=c, d=d,
                                beta=beta_value, n_steps=n_steps,
                                mode='moore',
                                record_every=2000)
        new_row = pl.DataFrame(
            {
                "scenario": [name],
                "probe_n": [probe_n],
                "cancer_frac_history": [hist['cancer_frac']]
            }
        )
        results_moore = pl.concat([results_moore, new_row], how='vertical')

### mode: vonneumann 
for name, (a, b, c, d) in scenarios.items():
    for probe_n in range(N_SIMS):
        _, hist = simulate(L=L, a=a, b=b, c=c, d=d,
                                beta=beta_value, n_steps=n_steps,
                                mode='vonneumann',
                                record_every=2000)
        new_row = pl.DataFrame(
            {
                "scenario": [name],
                "probe_n": [probe_n],
                "cancer_frac_history": [hist['cancer_frac']]
            }
        )
        results_vonneumann = pl.concat([results_vonneumann, new_row], how='vertical')




In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

## formatting part
combinded_dfs = []
dfs = {"moore": results_moore, "vonneumann": results_vonneumann}
for name, df in dfs.items():
    ### taking last results certain results
    df = df.with_columns(
        pl.col("cancer_frac_history").list.get(10, null_on_oob=True)
        .fill_null(pl.col('cancer_frac_history').list.last())
        .alias('cancer_frac_20k'),
        pl.col("cancer_frac_history").list.get(20, null_on_oob=True)
        .fill_null(pl.col('cancer_frac_history').list.last())
        .alias('cancer_frac_40k'),
    )
    ### adding conditions to count them
    df = df.with_columns(
        outcome_20k = pl.when(
            pl.col('cancer_frac_20k') == 1.0
        ).then(pl.lit("Cancer")).when(
            pl.col('cancer_frac_20k') == 0.0
        ).then(pl.lit("Immune")).otherwise(
            pl.lit('Coexistence')
        ),
        outcome_40k = pl.when(
            pl.col('cancer_frac_40k') == 1.0
        ).then(pl.lit("Cancer")).when(
            pl.col('cancer_frac_40k') == 0.0
        ).then(pl.lit("Immune")).otherwise(
            pl.lit('Coexistence')
        ),
        neighborhood = pl.lit(name)
    )
    combinded_dfs.append(df)
### joining dfs
df_all = pl.concat(combinded_dfs, how='vertical')
### retracting needed data
df_grouped = (
    df_all.select(['scenario', 'neighborhood', 'outcome_20k', 'outcome_40k'])
    .unpivot(
        index=['scenario', 'neighborhood'],
        on=['outcome_20k', 'outcome_40k'],
        variable_name='timepoint',
        value_name='outcome'
    ).with_columns(
        pl.col('timepoint').str.replace('outcome_', '')
    ).group_by(['scenario', 'neighborhood', 'timepoint', 'outcome'])
    .agg(
        pl.len().alias('count')
    )
)

## barplots - new version 
g = sns.catplot(
    data=df_grouped,
    kind="bar",
    x="scenario",
    y="count",
    hue="outcome",
    col="timepoint",      # Splits columns for 10k and 60k 
    row="neighborhood",   # Splits rows for moore and vonneneuman
    palette={"Cancer": "red", "Immune": "green", "Coexistence": "gray"},
    height=4,
    aspect=1.5,
    
)

# Kosmetyka wykresu
g.set_axis_labels("Scenario", "Number of Simulations")
g.set_titles("{row_name} model | Time: {col_name}")
plt.subplots_adjust(top=0.9)
g.fig.suptitle("Simulation Outcomes by Scenario, Timepoint, and Neighborhood")
plt.show()

In [ ]:
# histogram 
sns.histplot(
    data=df_all.filter(
    pl.col('scenario') == 'Coexistence'
    ),
    x='cancer_frac_20k',
    stat='proportion'
)


***

## Exercise 2 — Parameter sweep: how aggressive must Cancer be?

### Goal
Investigate how the parameter $b$ (Cancer benefit from Immune neighbours — a proxy
for *immune evasion*) affects outcome.

### Instructions

1. Fix $a=1.0,\; c=0.5,\; d=1.5,\; \beta=1.0$ and $L=25$.
2. Vary $b$ over the range $\{0.0, 0.25, 0.5, \ldots, 3.0\}$ (13 values).
3. For each value of $b$, run **30 simulations** and record:
   - the fraction of runs won by Cancer,
   - the **mean time to fixation** (number of steps until one type disappears;
     count simulations that timed-out as `n_steps`).
4. Produce **two plots** (side by side):
   - Left: Cancer winning probability vs $b$.
   - Right: Mean time to fixation vs $b$.
5. Mark the approximate **critical value** of $b$ at which Cancer starts to win
   more than 50 % of the time.

**Hint:** The `simulate` function returns `history["steps_done"]` which gives the
number of competition events that actually occurred.

> **Bonus:** Repeat for $\beta \in \{0.5, 1.0, 2.0\}$.  How does selection
> intensity shift the critical $b$ value?


### Answers
#### 5. 
$b = 1.25 $
#### Bonus
$beta$ parameters defines impact of difference in *fitness functions*. With small $beta$ parameter the model is more noisy. We conclude that from higher value of the *Mean steps statistic*. With higher $beta$ noise is much smaller and depends mostly on *fitness functions*. We can see by from much lower *Mean steps statistic*.   

$beta$ does not affect critical $b$ value, it only affects *convergence tempo* not *limit*. That is because in equation $(a, b, c, d)$ parameters affects critical points that is when condition:
$$f_I - f_C < 0 $$
is met. 



In [ ]:
# ─── Exercise 2: Your code here ───

b_values = np.arange(0, 3.25, 0.25)
beta_values = np.array([0.5, 1.0, 2.0])
a, c, d = 1.0, 0.5, 1.5
L = 25
N_SIMS = 30
n_steps = 100_000 # here everything should be resolved 

cancer_win_frac = []
mean_fixation_time = []

# --- YOUR CODE BELOW ---
## Simulation part
dfs = list()
### running for certain beta
for beta_value in beta_values:
    #### creating df for certain beta value 
    results_b = pl.DataFrame(
    {
        "beta_value": [],
        "b_value": [],
        "probe_n": [],
        "cancer_frac_history": [],
        "n_steps": []
    },
    schema={
        "beta_value": pl.Float64,
        "b_value": pl.Float64,
        "probe_n": pl.Int64,
        "cancer_frac_history": pl.Float64,
        "n_steps": pl.Int64
    })
    #### running tests for b_value 
    for b_value in b_values:
        for probe_n in range(N_SIMS):
            _, hist = simulate(L= L, a=a, b=b_value, c=c, d=d,
                                    beta=beta_value, n_steps=n_steps,
                                    mode='moore',
                                    record_every=2000)
            new_row = pl.DataFrame(
                {
                    "beta_value": [beta_value],
                    "b_value": [b_value],
                    "probe_n": [probe_n],
                    "cancer_frac_history": [hist['cancer_frac'][-1]],
                    "n_steps": [hist['steps_done']]
                }
            )
            results_b = pl.concat([results_b, new_row], how='vertical')
    #### appending df to dfs
    dfs.append(results_b)

### Merging all dfs
df = pl.concat(dfs, how="vertical")


In [ ]:
df

In [ ]:
# grouping values
df_agg = df.group_by(
    ['beta_value', 'b_value']
).agg(
    # probability of weaning 
    (pl.col('cancer_frac_history')==1.0).mean().alias('prob_cancer_win'),
    # mean time
    pl.col('n_steps').mean().alias('mean_fixation_time')
).sort(['beta_value', 'b_value'])
# df_agg

# plots
## create object
fig, axes = plt.subplots(1, 2, figsize = (14, 5))

## left plot
sns.lineplot(
    data=df_agg, 
    x='b_value', 
    y='prob_cancer_win', 
    hue='beta_value',   # splits line colors 
    marker='o',         # mark data points
    palette='tab10',
    ax=axes[0]          
)

axes[0].axhline(y=0.5, color='gray', linestyle='--', label='50% threshold')
axes[0].set_title('Cancer winning probability vs b')
axes[0].set_xlabel('Cancer benefit from Immune (b)')
axes[0].set_ylabel('Probability of Cancer winning')
axes[0].legend(title='Beta')

sns.lineplot(
    data=df_agg, 
    x='b_value', 
    y='mean_fixation_time', 
    hue='beta_value', 
    marker='o',
    palette='tab10',
    ax=axes[1]          
)
axes[1].set_title('Mean time to fixation vs b')
axes[1].set_xlabel('Cancer benefit from Immune (b)')
axes[1].set_ylabel('Mean steps')
axes[1].legend(title='Beta')

plt.tight_layout()
plt.show()

---

## Exercise 3 — Visualising spatial dynamics

### Goal
Produce **animated snapshots** of the lattice over time and relate spatial patterns
to the underlying parameters.

### Instructions

1. Run the simulation for *each* of the three scenarios ($L=40$, up to 100 000
   steps) and record snapshots every 2 000 steps (use `record_every=2000`).

2. For each scenario, display a row of **6 evenly-spaced snapshots** (use the code
   pattern from the demo above) so you can see how the spatial pattern evolves.

3. For the **Coexistence** scenario, additionally create an **animation** (GIF or
   inline HTML video) showing the grid at every recorded snapshot.
   
   A helper function is provided below — you only need to pass it the list of
   snapshots.

4. In a Markdown cell, write **3–5 sentences** describing the qualitative differences in spatial dynamics between the three scenarios. Consider:
   - Does one cell type form compact clusters?
   - How fast does spatial segregation develop?
   - Is the boundary between the two types smooth or fractal-like?


### Answers

#### 4. 
From previous answers we can conclude all considerations
* Does one cell type form compact clusters?
   - no, only amount of neighbors impact behavior of model (same equations as in exercise 2)
* How fast does spatial segregation develop?
   - it depends on $\beta$ parameter
* Is the boundary between the two types smooth or fractal-like?
   - fractal like


In [ ]:
# Helper: create an inline animation from a list of grid snapshots

def animate_snapshots(snapshots, record_every=2000, interval=200):
    """
    Create a matplotlib animation from a list of 2-D grid arrays.
    Returns an HTML object you can display in Jupyter with display(anim).
    
    Parameters
    ----------
    snapshots : list of 2-D numpy arrays
    interval  : int, milliseconds between frames
    """
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(snapshots[0], cmap=CMAP, norm=NORM, interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([])
    title = ax.set_title("step 0")

    def update(frame):
        im.set_data(snapshots[frame])
        title.set_text(f"step {frame * record_every}")
        return [im, title]

    anim = animation.FuncAnimation(fig, update, frames=len(snapshots),
                                   interval=interval, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


In [ ]:
# ─── Exercise 3: Your code here ───

scenarios = {
    "Cancer advantage": (1.0, 2.0, 0.0, 1.0),
    "Coexistence":      (0.8, 1.2, 0.5, 1.5),
    "Immunotherapy":    (0.6, 0.5, 2.0, 1.8),
}

L = 40
n_steps = 100_000
record_every = 2000
beta_value = 1.0

# --- YOUR CODE BELOW ---

all_snapshots = {}

results_moore = pl.DataFrame(
    {
        "scenario": [],
        "probe_n": [],
        "cancer_frac_history": []
    },
    schema={
        "scenario": pl.String,
        "probe_n": pl.Int64,
        "cancer_frac_history": pl.List(pl.Float64)
    }
)

for name, (a, b, c, d) in scenarios.items():
    print(f"Running scenario: {name}...")
    
    _, hist = simulate(L=L, a=a, b=b, c=c, d=d,
                       beta=beta_value, n_steps=n_steps,
                       mode='moore',
                       record_every=record_every)
    
    snapshots = hist['snapshots']
    all_snapshots[name] = snapshots
    
    n_snaps = len(snapshots) # IMPORTANT - we simulations end earlier
    indices = np.linspace(0, n_snaps - 1, 6, dtype=int)
    
    fig, axes = plt.subplots(1, 6, figsize=(18, 3))
    for ax, idx in zip(axes, indices):
        step_label = idx * record_every 
        show_grid(snapshots[idx], title=f"step {step_label}", ax=ax)
        
    plt.suptitle(f"Scenario: {name}", y=1.05, fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
animate_snapshots(hist['snapshots'])